In [ ]:
"""Viabilidad LSTM -- SAPI-63 Analisis Exploratorio de Secuencias Temporales.

OBJETIVO: evaluar si la estructura del dataset temporal permite construir
secuencias causales por celda, cuantificar el desbalance real y concluir
sobre la viabilidad de una arquitectura LSTM para este dominio.

AVISO: este notebook es EXPLORATORIO. No entrena LSTM. No fabrica metricas.
No modifica Modelo D. Conforme a ADR-006.

REPRODUCIBILIDAD: requiere data/processed/temporal_dataset_h6.parquet
(generado por scripts/build_temporal_dataset.py con datos reales).
"""
# -*- coding: utf-8 -*-

# # SAPI-63 -- Viabilidad LSTM: Secuencias Temporales Causales
#
# **Proposito:** caracterizar el dataset temporal para evaluar si una
# arquitectura secuencial (LSTM/RNN) es viable dado el volumen de datos,
# el desbalance clase y la estructura espacio-temporal.
#
# **No se entrena ningun modelo. No se modifican candidatos del CORE.**

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Asegura que el path del repo sea reconocible
REPO_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT))

from experiments.sapi63.temporal_sequences import (
    N_FEATURES,
    SEQUENCE_FEATURE_COLS,
    _cell_sequences,
    build_sequence_folds,
)

PARQUET_PATH = REPO_ROOT / "data" / "processed" / "temporal_dataset_h6.parquet"

if not PARQUET_PATH.exists():
    raise FileNotFoundError(
        "Dataset no encontrado en " + str(PARQUET_PATH) + ". "
        "Ejecutar scripts/build_temporal_dataset.py primero."
    )

# ## 1. Carga y estadisticas base del dataset flat

In [ ]:
df = pd.read_parquet(PARQUET_PATH)
eligible = df[df["eligible_for_training"]].copy()

print("=== DATASET FLAT (todas las filas) ===")
print("  Total filas:              " + str(len(df)))
print("  Filas elegibles:          " + str(len(eligible)))
print("  Filas excluidas (cooldown): " + str(len(df) - len(eligible)))
print("  Celdas:                   " + str(df['cell_id'].nunique()))
print("  Timestamps unicos:        " + str(df['forecast_time'].nunique()))
print("  Periodo:                  " + str(df['forecast_time'].min()) + " -> " + str(df['forecast_time'].max()))

# ## 2. Distribucion de positivos y desbalance real

In [ ]:
n_pos = int((eligible["target"] == 1).sum())
n_neg = int((eligible["target"] == 0).sum())
n_total = n_pos + n_neg
ratio = n_neg / n_pos if n_pos > 0 else float("inf")

print("=== DESBALANCE CLASE (filas elegibles) ===")
print("  Positivos (target=1):   " + str(n_pos))
print("  Negativos (target=0):   " + str(n_neg))
print("  Total:                  " + str(n_total))
print("  Razon negativo:positivo = " + str(int(ratio)) + ":1")
print("  Tasa positiva:          " + str(round(n_pos / n_total * 100, 4)) + "%")
print()
print("AVISO CIENTIFICO: el score no es una probabilidad calibrada.")
print("Esta razon (1:" + str(int(ratio)) + ") no es una ley universal -- es la")
print("distribucion real del dataset con el periodo y umbrales actuales.")

# ## 3. Distribucion de positivos por celda

In [ ]:
pos_por_celda = eligible[eligible["target"] == 1].groupby("cell_id").size()
n_celdas_con_pos = len(pos_por_celda)
n_celdas_total = eligible['cell_id'].nunique()
n_celdas_sin_pos = n_celdas_total - n_celdas_con_pos
celdas_le2 = int((pos_por_celda <= 2).sum())
celdas_le5 = int((pos_por_celda <= 5).sum())

print("=== POSITIVOS POR CELDA ===")
print("  Celdas con >=1 positivo: " + str(n_celdas_con_pos) + " de " + str(n_celdas_total))
print("  Celdas sin ningun positivo: " + str(n_celdas_sin_pos))
print()
print("Distribucion de positivos en celdas con >=1 evento:")
print(pos_por_celda.describe().to_string())
print()
print("  Celdas con <=2 eventos (evidencia MUY escasa para LSTM): " + str(celdas_le2))
print("  Celdas con <=5 eventos: " + str(celdas_le5))

# ## 4. Frecuencia temporal y continuidad de la serie

In [ ]:
# Verificar la frecuencia de muestreo
times = pd.Series(eligible["forecast_time"].unique()).sort_values()
gaps = times.diff().dropna()
n_gaps_gt24h = int((gaps > pd.Timedelta(hours=24)).sum())

print("=== CONTINUIDAD TEMPORAL ===")
print("  Frecuencia nominal: 6h (por CANDIDATE_STEP_HOURS en build_temporal_dataset.py)")
print("  Gap mediano real:   " + str(gaps.median()))
print("  Gap maximo:         " + str(gaps.max()))
print("  Pasos con gap > 24h: " + str(n_gaps_gt24h))
print()
print("Implicacion para LSTM: gaps > 6h rompen la asuncion de continuidad")
print("de la serie temporal. Las ventanas con saltos temporales son descartadas.")

# ## 5. Construccion de secuencias -- shape 3D por celda

In [ ]:
# Construimos folds con window_size=4 (24h de contexto @ 6h/paso)
WINDOW_SIZE = 4

print("=== CONSTRUCCION DE SECUENCIAS (window_size=" + str(WINDOW_SIZE) + ") ===")
print("  Features incluidas: " + str(N_FEATURES))
print("  Features: " + str(SEQUENCE_FEATURE_COLS))

# Estadisticas globales de secuencias y descartes sobre el dataset elegible
total_insuf_global = 0
total_gap_global = 0
total_valid_global = 0
pos_valid_global = 0

for cell_id, cell_df in eligible.groupby("cell_id"):
    X_cell, y_cell, n_insuf, n_gap = _cell_sequences(
        cell_df, window_size=WINDOW_SIZE, feature_cols=SEQUENCE_FEATURE_COLS
    )
    total_insuf_global += n_insuf
    total_gap_global += n_gap
    total_valid_global += len(X_cell)
    pos_valid_global += int(y_cell.sum())

pos_gap_global = n_pos - pos_valid_global

print()
print("=== METRICAS GLOBALES DE SECUENCIAS 3D Y CONTINUIDAD ===")
print("  Filas elegibles:                    " + str(len(eligible)))
print("  Secuencias 3D validas generadas:    " + str(total_valid_global))
print("  Shape 3D global:                    (" + str(total_valid_global) + ", " + str(WINDOW_SIZE) + ", " + str(N_FEATURES) + ")")
print("  Ventanas descartadas (insuficiente):" + str(total_insuf_global) + " (pasos iniciales)")
print("  Ventanas descartadas (salto > 6h):  " + str(total_gap_global) + " (discontinuidad temporal)")
print("  Positivos en secuencias validas:    " + str(pos_valid_global) + " de " + str(n_pos))
print("  Positivos descartados por salto:    " + str(pos_gap_global))

folds = build_sequence_folds(
    df,
    window_size=WINDOW_SIZE,
    min_train_months=12,
)

print()
print("  Total folds walk-forward generados: " + str(len(folds)))

# ## 6. Distribucion real de positivos por fold (sin fabricar)

In [ ]:
print("=== DISTRIBUCION REAL DE POSITIVOS Y DESCARTE POR FOLD ===")
print("{:>4} {:>20} {:>18} {:>10} {:>10} {:>10} {:>12} {:>12}".format(
    "Fold", "Train shape", "Test shape",
    "Pos_train", "Pos_test", "Disc_gap", "Rate_test", "Ratio_test"
))
print("-" * 120)

fold_summaries = []
for fold in folds:
    s = fold.summary()
    fold_summaries.append(s)
    ratio_test_val = (
        str(int((1 - fold.positive_rate_test) / fold.positive_rate_test))
        if fold.positive_rate_test > 0
        else "inf"
    )
    print("{:>4} {:>20} {:>18} {:>10} {:>10} {:>10} {:>12.6f} {:>12}".format(
        fold.fold_index,
        str(fold.X_train.shape),
        str(fold.X_test.shape),
        fold.n_positives_train,
        fold.n_positives_test,
        fold.n_discarded_discontinuity,
        fold.positive_rate_test,
        ratio_test_val,
    ))

print()
total_pos_test = sum(f.n_positives_test for f in folds)
print("Total positivos en test (suma de todos los folds): " + str(total_pos_test))
print("Folds con 0 positivos en test: " + str(sum(1 for f in folds if f.n_positives_test == 0)))

# ## 7. Verificacion de causalidad de las secuencias

In [ ]:
print("=== VERIFICACION DE CAUSALIDAD ===")
print("  La ventana [T-w, T) contiene unicamente pasos anteriores a T.")
print("  El target de la muestra corresponde al paso T (no a la ventana).")
print()

# Verificacion directa en el primer fold con datos
folds_with_data = [f for f in folds if len(f.X_train) > 0 and len(f.X_test) > 0]
if folds_with_data:
    fold = folds_with_data[0]
    print("  Fold " + str(fold.fold_index) + " -- X_train shape: " + str(fold.X_train.shape))
    print("  Periodo train: " + str(fold.train_period[0]) + " -> " + str(fold.train_period[1]))
    print("  Periodo test:  " + str(fold.test_period[0]) + " -> " + str(fold.test_period[1]))
    assert fold.train_period[1] < fold.test_period[0], "ERROR: solapamiento temporal detectado"
    print("  PASS: Causalidad walk-forward verificada en este fold")
else:
    print("  No hay folds con datos suficientes para verificar.")

# ## 8. Conclusion fundamentada sobre viabilidad LSTM y desbalance 1:3393

In [ ]:
print("=" * 70)
print("CONCLUSION -- VIABILIDAD LSTM / RNN PARA SAPI (SAPI-63)")
print("=" * 70)

conclusion_dict = {
    "dataset_shape_flat": str(len(eligible)) + " filas elegibles",
    "n_cells": int(eligible["cell_id"].nunique()),
    "n_timestamps": int(eligible["forecast_time"].nunique()),
    "n_positivos": n_pos,
    "n_negativos": n_neg,
    "ratio_desbalance": "1:" + str(int(ratio)),
    "n_features_secuencia": N_FEATURES,
    "window_size_usado": WINDOW_SIZE,
    "contexto_24h": True,
    "n_folds_walk_forward": len(folds),
    "folds_con_cero_positivos_test": sum(1 for f in folds if f.n_positives_test == 0),
    "secuencias_validas_totales": total_valid_global,
    "shape_3d_secuencias": [total_valid_global, WINDOW_SIZE, N_FEATURES],
    "ventanas_descartadas_insuficiente": total_insuf_global,
    "ventanas_descartadas_discontinuidad": total_gap_global,
    "positivos_en_secuencias_validas": pos_valid_global,
    "positivos_descartados_discontinuidad": pos_gap_global,
}

print()
print("FACTORES FAVORABLES:")
print("  OK Estructura regular: grilla celda x tiempo, 50 celdas x " + str(df['forecast_time'].nunique()) + " timestamps.")
print("  OK Secuencias 3D disponibles sin reshaping estructural.")
print("  OK " + str(len(folds)) + " folds walk-forward generados con separacion causal real.")
print("  OK Features de causalidad verificada (ninguna feature > T en la ventana).")
print("  OK Flagging explicito de NaN -- sin imputacion disfrazada.")
print()
print("FACTORES EN CONTRA / LIMITACIONES:")
print("  XX DESBALANCE EXTREMO: 1:" + str(int(ratio)) + " negativo:positivo.")
print("     " + str(n_pos) + " positivos en " + str(n_total) + " filas elegibles (" + str(round(n_pos/n_total*100, 4)) + "%).")
print("     Una LSTM que predice siempre 0 tiene accuracy >99.97% -- metrica inutil.")
print("     Se requiere PR-AUC, recall a precision fija, calibracion.")
print()
print("  XX POSITIVOS ESCASOS POR CELDA:")
print("     " + str(celdas_le2) + " celdas tienen <=2 eventos historicos.")
print("     Una LSTM necesita secuencias positivas en train para aprender la")
print("     representacion del evento -- con 0-2 ejemplos, el gradiente no puede")
print("     generalizarse a nuevas manifestaciones.")
print()
print("  XX CONCENTRACION TEMPORAL (megaevento 2024-02-03):")
print("     21.5% de los positivos estan en un solo dia (23 de 107).")
print("     Cualquier fold que cruce esa fecha reporta resultados no generalizables.")
print()
print("  XX LAGUNAS EN LA SERIE TEMPORAL:")
print("     Gap maximo entre timestamps: " + str(gaps.max()) + ".")
print("     Una LSTM asume continuidad; las lagunas introducen discontinuidades")
print("     que el modelo interpreta como parte de la secuencia, no como ausencia.")
print()
print("VEREDICTO (alineado con ADR-006 y freeze de arquitectura Sprint 2):")
print()
print("  El dataset TIENE la estructura minima para construir secuencias")
print("  temporales causales (shape 3D verificado, walk-forward limpio).")
print()
print("  Sin embargo, con " + str(n_pos) + " positivos repartidos en " + str(n_celdas_total) + " celdas, de los")
print("  cuales la mayoria de celdas tienen <=2 eventos, y con un megaevento")
print("  que representa >21% de los positivos en un solo dia, entrenar y")
print("  VALIDAR una LSTM con confianza estadistica es inviable con los")
print("  datos actuales.")
print()
print("  RECOMENDACION: mantener Modelo D como baseline (ADR-006). Una")
print("  LSTM requiere al menos 5-10x mas positivos por celda para ser")
print("  comparable. El modulo experiments/sapi63/temporal_sequences.py queda disponible")
print("  como infraestructura para cuando el volumen de datos lo permita.")
print()
print("  Esta conclusion no es definitiva: es funcion del dataset actual.")
print("  Revisar si el backfill FIRMS/DMC crece en cobertura temporal.")
print("=" * 70)

In [ ]:
# Guardar resumen en JSON para trazabilidad
output_path = REPO_ROOT / "reports" / "sapi63_lstm_viability_report.json"
output_path.parent.mkdir(exist_ok=True)

report = {
    "sapi_issue": "SAPI-63",
    "titulo": "Viabilidad LSTM -- Secuencias Temporales Causales",
    "dataset": str(PARQUET_PATH),
    "estadisticas": conclusion_dict,
    "fold_summaries": fold_summaries,
    "conclusion": {
        "estructura_3d_disponible": True,
        "walk_forward_causal_limpio": True,
        "n_positivos_insuficiente_para_lstm": True,
        "megaevento_concentrado": True,
        "lagunas_temporales_detectadas": n_gaps_gt24h,
        "ventanas_descartadas_discontinuidad": total_gap_global,
        "secuencias_validas_totales": total_valid_global,
        "positivos_en_secuencias_validas": pos_valid_global,
        "veredicto": (
            "INVIABLE con datos actuales (" + str(n_pos) + " positivos flat, "
            + str(pos_valid_global) + " en secuencias causales continuas; "
            + "mayoria de celdas con <=2 eventos). "
            "Modelo D permanece baseline (ADR-006). Infraestructura de secuencias disponible."
        ),
        "metricas_requeridas_si_se_entrena": ["PR-AUC", "recall@precision_fija", "calibracion"],
        "aviso": "Score no es probabilidad calibrada. No generalizar sin evidencia.",
    },
}

output_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
print()
print("Reporte guardado en: " + str(output_path))
print("LISTO -- notebook reproducible completado.")